# Churn prediction for selected players

Set `PLAYER_IDS` (one id or a list) and `SNAPSHOT` in the parameters cell and run the
notebook. For every requested player who is at risk of churn on the snapshot date it
shows:

- the scorecard: churn probability within 7, 14, 30 and 60 days, expected days until
  churn, risk segment, and the rank in the whole base at risk;
- the survival curve against the average of the base;
- the three features that push the risk up the most;
- the feature values the prediction was made from.

A requested player who cannot be scored is listed with the reason. Nothing is trained
and nothing is written to disk: the notebook only reads `data/` and `artifacts/`, so run
`python -m churn.generate_data` and `python -m churn.pipeline` first.

In [1]:
import warnings

# Optuna imports tqdm.auto, which warns when ipywidgets is missing; no progress bar is used.
warnings.filterwarnings("ignore", message="IProgress not found")

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.colors import qualitative

from churn import config, features
from churn import evaluation as ev
from churn.models import MODELS, SurvivalModel
from churn.pipeline import score_players

DATA_DIR = config.DATA_DIR
ARTIFACTS_DIR = config.ARTIFACTS_DIR
MODELS_DIR = ARTIFACTS_DIR / "models"
TIMES = config.TIME_GRID  # days 1..85 on which survival curves are evaluated
HORIZON = config.HORIZON_DAYS
FEATURES = list(config.FEATURES)

pd.set_option("display.precision", 3, "display.max_columns", 40, "display.max_colwidth", 120)

if not (ARTIFACTS_DIR / "metrics.json").exists() or not (DATA_DIR / "activity.parquet").exists():
    raise FileNotFoundError(
        "data/ or artifacts/ is missing; run `uv run python -m churn.generate_data` "
        "and `uv run python -m churn.pipeline` first"
    )

## Parameters

In [3]:
# One player id or a list of ids. Player 1 churned in 2024 and shows how a player who
# cannot be scored is reported.
PLAYER_IDS = [19264, 18098, 19032, 13237, 1]

# Date of the prediction. Only activity strictly before it is used. The default is the day
# after the last observed day, i.e. the real scoring date; any date inside the data works.
SNAPSHOT = config.SCORING_SNAPSHOT

## Load the model and the data

In [4]:
def load_model(name: str) -> SurvivalModel:
    """Load a fitted model; its class is recorded in the meta file."""
    prefix = MODELS_DIR / name
    meta = json.loads(Path(f"{prefix}.meta.json").read_text())
    return MODELS[meta["model"]].load(prefix)


def load_calibrator(name: str) -> ev.IsotonicSurvivalCalibrator:
    """Load the calibrator saved as ``calibrator_<name>.json``."""
    data = json.loads((MODELS_DIR / f"calibrator_{name}.json").read_text())
    return ev.IsotonicSurvivalCalibrator.from_dict(data)


best_variant = json.loads((ARTIFACTS_DIR / "metrics.json").read_text())["best_variant"]
production = load_model("production")
# The pipeline applies the calibrator only if model selection chose the calibrated variant.
calibrator = load_calibrator("production") if best_variant == "calibrated" else None
players, activity = features.load_raw(DATA_DIR)
churn_dates = features.first_churn_dates(activity, config.DATA_END)
print(f"model: {type(production).__name__} ({production.num_boost_round} trees), {best_variant}")
print(
    f"data: {len(players):,} players, activity {config.DATA_START.date()}..{config.DATA_END.date()}"
)

model: CoxModel (275 trees), calibrated
data: 20,000 players, activity 2024-01-01..2026-06-30


## Check the request

The snapshot must lie inside the data. A player is scored if they are at risk on the
snapshot date: they have played before it and have not churned yet.

In [5]:
snapshot = pd.Timestamp(SNAPSHOT).normalize()
first_snapshot = config.DATA_START + pd.Timedelta(days=1)
last_snapshot = config.DATA_END + pd.Timedelta(days=1)
if not first_snapshot <= snapshot <= last_snapshot:
    raise ValueError(
        f"SNAPSHOT {snapshot.date()} is outside the data: "
        f"choose a date from {first_snapshot.date()} to {last_snapshot.date()}"
    )
if snapshot < config.SCORING_SNAPSHOT:
    print(
        f"note: the production model was trained on outcomes observed up to "
        f"{config.SCORING_SNAPSHOT.date()}, so a prediction for {snapshot.date()} "
        "is not out-of-sample\n"
    )
requested = list(dict.fromkeys(int(player_id) for player_id in np.atleast_1d(PLAYER_IDS)))
if not requested:
    raise ValueError("PLAYER_IDS is empty")

frame = features.snapshot_features(players, activity, churn_dates, snapshot)
at_risk = set(frame["player_id"])
known = players.set_index("player_id")
last_active = activity[activity["date"] < snapshot].groupby("player_id")["date"].max()


def why_not_scored(player_id: int) -> str:
    """Reason why a player has no prediction on the snapshot date."""
    if player_id not in known.index:
        return "unknown player_id"
    if player_id not in last_active.index:
        registered = known.loc[player_id, "registration_date"].date()
        return f"no activity before the snapshot (registered on {registered})"
    if player_id in churn_dates.index and churn_dates[player_id] < snapshot:
        last_day = last_active[player_id].date()
        return f"already churned on {churn_dates[player_id].date()} (last active on {last_day})"
    raise RuntimeError(f"player {player_id} should be at risk but has no feature row")


scored_ids = [player_id for player_id in requested if player_id in at_risk]
skipped = {
    player_id: why_not_scored(player_id) for player_id in requested if player_id not in at_risk
}
print(f"snapshot {snapshot.date()}: {len(frame):,} players at risk")
print(f"requested {len(requested)}, scored {len(scored_ids)}, not scored {len(skipped)}")
for player_id, reason in skipped.items():
    print(f"  player {player_id}: {reason}")
if not scored_ids:
    raise ValueError("none of the requested players is at risk on the snapshot date")

snapshot 2026-07-01: 4,328 players at risk
requested 5, scored 4, not scored 1
  player 1: already churned on 2024-05-14 (last active on 2024-04-14)


## Score

The whole base at risk is scored and the requested players are then picked from it, so
`risk_rank` is the place of a player among everybody at risk on the snapshot date
(1 = the riskiest), and `top_percent` is the share of the base ranked at least as high.

In [6]:
scores, curves = score_players(production, calibrator, frame)
scores["top_percent"] = 100 * scores["risk_rank"] / len(scores)
# Churn is 30 inactive days in a row, so it cannot happen before this day.
scores["earliest_churn_day"] = config.INACTIVITY_DAYS + 1 - scores["recency_days"]

rows = scores.index[scores["player_id"].isin(scored_ids)]
selected = scores.loc[rows].set_index("player_id").loc[scored_ids]
selected_curves = curves.loc[rows].set_axis(scores.loc[rows, "player_id"]).loc[scored_ids]
print(f"scored {len(scores):,} players, selected {len(selected)}")

if snapshot == config.SCORING_SNAPSHOT:
    # On the scoring date the result must equal what the pipeline saved.
    saved = pd.read_parquet(ARTIFACTS_DIR / "scores.parquet").set_index("player_id")
    saved = saved.loc[scores["player_id"]]
    assert (saved["risk_rank"].to_numpy() == scores["risk_rank"].to_numpy()).all()
    for column in [*[f"churn_prob_{h}d" for h in config.SCORE_HORIZONS], "rul_days", "risk_score"]:
        assert np.allclose(saved[column], scores[column]), column
    print("OK: equals artifacts/scores.parquet for the whole base")

scored 4,328 players, selected 4
OK: equals artifacts/scores.parquet for the whole base


## Scorecard

- `churn_prob_<h>d`: probability of churning within `h` days of the snapshot, calibrated
  if model selection chose the calibrated variant.
- `rul_days`: expected days until churn, capped at 85.
- `risk_segment`: `high` from 0.70, `medium` from 0.40 on the 60-day probability.
- `earliest_churn_day`: `31 - recency_days`, the first day on which the player can churn
  by definition. The model starts its clock on that day, so the probabilities for
  shorter horizons are exactly zero.

In [7]:
card_columns = ["risk_rank", "top_percent", "risk_segment"]
card_columns += [f"churn_prob_{h}d" for h in config.SCORE_HORIZONS]
card_columns += ["rul_days", "recency_days", "earliest_churn_day"]
display(selected[card_columns])

,risk_rank,top_percent,risk_segment,churn_prob_7d,churn_prob_14d,churn_prob_30d,churn_prob_60d,rul_days,recency_days,earliest_churn_day
player_id,,,,,,,,,,
19264,439,10.143,high,0.0,0.0,0.495,0.778,42.188,4,27
18098,504,11.645,medium,0.0,0.0,0.495,0.646,48.824,4,27
19032,905,20.910,low,0.0,0.0,0.130,0.315,67.530,3,28
13237,12,0.277,high,1.0,1.0,1.000,1.000,1.000,30,1


## Survival curves

Probability that the player has not churned by each day after the snapshot. The dashed
line is the average over the whole base at risk.

In [8]:
days = [0, *TIMES]
fig = go.Figure()
for number, (player_id, curve) in enumerate(selected_curves.iterrows()):
    label = f"player {player_id} ({selected.loc[player_id, 'risk_segment']})"
    color = qualitative.Plotly[number % len(qualitative.Plotly)]
    fig.add_trace(go.Scatter(x=days, y=[1.0, *curve], name=label, line={"color": color}))
average = [1.0, *curves.mean()]
style = {"color": "#6b7280", "dash": "dash"}
fig.add_trace(go.Scatter(x=days, y=average, name=f"average of {len(curves):,} at risk", line=style))
fig.add_vline(x=HORIZON, line_dash="dot", line_color="#6b7280")
fig.update_layout(
    title=f"Survival curves on {snapshot.date()}",
    xaxis_title="days since the snapshot",
    yaxis_title="probability of not having churned",
    yaxis_tickformat=".0%",
    yaxis_range=[0, 1.02],
    template="plotly_white",
    height=440,
)
fig.show()

## Risk drivers

The three features with the largest positive SHAP contribution to the player's risk
score, with the value the player has. `shap` is on the scale of the risk score (for the
Cox model the log hazard ratio): a value of 1 multiplies the hazard by `e`. The risk
score describes the time left after `earliest_churn_day`; the wait until that day is
fixed by `recency_days` and is not part of it.

In [9]:
drivers = pd.DataFrame(
    [
        {
            "player_id": player_id,
            "driver": rank,
            "feature": card[f"driver_{rank}"],
            "value": card[card[f"driver_{rank}"]],
            "shap": card[f"driver_{rank}_shap"],
        }
        for player_id, card in selected.iterrows()
        for rank in (1, 2, 3)
    ]
)
display(drivers.set_index(["player_id", "driver"]))

feature     value   shap
player_id driver                                        
19264     1              active_days_7d         1  0.503
          2       activity_ratio_7d_30d     0.214  0.471
          3         acquisition_channel  paid_ads  0.268
18098     1             active_days_30d         4  0.419
          2                sessions_30d         9  0.272
          3         acquisition_channel  paid_ads  0.255
19032     1         acquisition_channel  paid_ads  0.319
          2             active_days_30d        10  0.198
          3                     ggr_30d     712.6  0.140
13237     1                recency_days        30  1.081
          2              active_days_7d         0  0.431
          3       activity_ratio_7d_30d       0.0  0.371

## Feature values

What the model was given for each player, next to the typical value in the base at risk
(median for numeric features, the most frequent value for categorical ones).

In [10]:
base = scores[FEATURES]
typical = {
    name: base[name].mode()[0] if name in config.CATEGORICAL_FEATURES else base[name].median()
    for name in FEATURES
}
table = selected[FEATURES].T
table.columns = [f"player {player_id}" for player_id in table.columns]
table["typical in the base"] = pd.Series(typical)
display(table)

,player 19264,player 18098,player 19032,player 13237,typical in the base
tenure_days,29,8,62,55,161.0
recency_days,4,4,3,30,2.0
active_days_7d,1,3,3,0,4.0
active_days_30d,20,4,10,1,15.0
activity_ratio_7d_30d,0.214,3.214,1.286,0.0,0.989
sessions_30d,43,9,21,1,30.0
turnover_30d,2175.59,454.57,4037.89,26.55,1732.535
ggr_30d,233.46,21.05,712.6,7.2,34.965
deposits_30d,141.78,152.81,826.78,19.09,259.64
withdrawals_30d,25.89,0.0,34.17,0.0,24.99


## Caveats

- **Short horizons.** Churn takes 30 inactive days, so the model counts time from
  `earliest_churn_day`: a curve stays at 100% until that day and no probability is placed
  on days when churn is impossible.
- **Past snapshots.** The production model was trained on the monthly snapshots before
  `config.SCORING_SNAPSHOT` with outcomes observed up to that date. Predictions for
  earlier dates are therefore not out-of-sample and look better than a real forecast
  would; the notebook prints a note when this applies.
- **Calibration.** The calibrator belongs to the model fitted `config.CALIBRATION_LAG`
  earlier, the latest one whose snapshot has full follow-up, and is applied only if model
  selection found that it helps (`best_variant` in `metrics.json`). Calibrated
  probabilities are step-like because the calibrator is an isotonic regression: several
  players can share exactly the same probability, and `risk_rank` then orders them by
  the raw probability.